In [ ]:
import pandas as pd

In [ ]:
import os
os.chdir(r"C:\Users\HP\OneDrive\Documents\Data analytics\India-Digital-Spending-Tax-Health-Monitor")
from src.config import NDAP_DIR, NDAP_FILE
from src.clean.state_utils import normalize_name

g = pd.read_csv(NDAP_DIR / NDAP_FILE)

In [ ]:
g.columns.tolist()

In [ ]:
print(g.isna().sum(min_count=1))

In [ ]:
print(g.isna().sum())

In [ ]:
null_mask=g.isna().any(axis=1)

In [ ]:
null_rows = g[null_mask]

In [ ]:
null_rows['State'].value_counts()

In [ ]:
null_rows['Year'].value_counts()

In [ ]:
state_percentages = null_rows['State'].value_counts(normalize=True) * 100
print("\nPercentage of nulls per state:\n", state_percentages)

In [ ]:
g

In [ ]:
null_rows['Month'].value_counts()

In [ ]:
g.shape

In [ ]:
# target_states = ["Daman and Diu", "Ladakh", "Other Territory"]
target_states = ["Other Territory"]
target_columns = [
    "Number Of Tax Payers Eligible To File (UOM:Number), Scaling Factor:1",
    "Number Of Tax Payers Filed By Due Date (UOM:Number), Scaling Factor:1",
    "Number Of Tax Payers Filed After Due Date (UOM:Number), Scaling Factor:1",
    "Total Returns Filed (UOM:Number), Scaling Factor:1",
    "Filing Percentage (UOM:%(Percentage)), Scaling Factor:1"
]
state_mask = g["State"].isin(target_states)
null_mask = g[target_columns].isnull().any(axis=1)
null_rows_in_states = g[state_mask & null_mask]
result = null_rows_in_states[["State", "Year", "Month"]].drop_duplicates().sort_values(by=["State", "Year", "Month"])
print("Months containing null values for the target states:\n", result)

In [ ]:

cols = [
    "Number Of Tax Payers Eligible To File (UOM:Number), Scaling Factor:1", 
    "Number Of Tax Payers Filed By Due Date (UOM:Number), Scaling Factor:1", 
    "Number Of Tax Payers Filed After Due Date (UOM:Number), Scaling Factor:1", 
    "Total Returns Filed (UOM:Number), Scaling Factor:1", 
    "Filing Percentage (UOM:%(Percentage)), Scaling Factor:1"
]

# Extract the strings directly to see what we are dealing with
ladakh_nulls = g[(g["State"] == "Ladakh") & (g[cols].isnull().any(axis=1))]

# Print the raw rows to see how the years and months are formatted
print("Raw columns for Ladakh nulls:")
print(ladakh_nulls[["Year", "Month"]].drop_duplicates())

# Safe datetime conversion
date_strings = ladakh_nulls["Month"].astype(str) + ", " + ladakh_nulls["Year"].astype(str)
ladakh_dates = pd.to_datetime(date_strings, format="%B, %Y", errors='coerce')

# Drop NaT values before finding min/max to avoid the strftime crash
valid_dates = ladakh_dates.dropna()

if not valid_dates.empty:
    print(f"Minimum Month: {valid_dates.min().strftime('%B %Y')}")
    print(f"Maximum Month: {valid_dates.max().strftime('%B %Y')}")
else:
    print("\n[Notice] No standard calendar months (like January, February) found in the null rows.")
    print("The dates are stored entirely as Financial Year text blocks.")


In [ ]:


# Define null columns
cols = [
    "Number Of Tax Payers Eligible To File (UOM:Number), Scaling Factor:1", 
    "Number Of Tax Payers Filed By Due Date (UOM:Number), Scaling Factor:1", 
    "Number Of Tax Payers Filed After Due Date (UOM:Number), Scaling Factor:1", 
    "Total Returns Filed (UOM:Number), Scaling Factor:1", 
    "Filing Percentage (UOM:%(Percentage)), Scaling Factor:1"
]

# Isolate Ladakh rows that contain null values
ladakh_nulls = g[(g["State"] == "Ladakh") & (g[cols].isnull().any(axis=1))].copy()

# 1. Convert the 'Month' column directly to datetime
ladakh_dates = pd.to_datetime(ladakh_nulls["Month"], format="%B, %Y")

# Print the Min and Max Dates
print(f"Minimum Month: {ladakh_dates.min().strftime('%B %Y')}")
print(f"Maximum Month: {ladakh_dates.max().strftime('%B %Y')}\n")

# 2. Filter and print Ladakh rows for January 2020
print("--- Ladakh Rows for January 2020 ---")
jan_2020_rows = g[(g["State"] == "Ladakh") & (g["Month"] == "January, 2020")]
print(jan_2020_rows)


In [ ]:
g.columns = ["country", "fy", "month", "state_code", "state", "return_type", "info",
             "eligible", "on_time", "late", "total_filed", "filing_pct"]

print(g.columns.tolist())

In [ ]:
strange_rows_count = ((g["eligible"] == 0) & (g["total_filed"] > 0)).sum()


In [ ]:
print(strange_rows_count)

In [ ]:
g[(g["eligible"] == 0) & (g["total_filed"] > 0)]

In [ ]:
from src.clean.state_utils import normalize_name, apply_mapping

g_copy = g.copy()
g_copy["state"] = g_copy["state"].apply(normalize_name)
g_copy = apply_mapping(g_copy, "ndap")
# g_copy.head()
print(g_copy["state"].value_counts().head())
print(g_copy["state"].value_counts().tail())

In [ ]:
mask = bengol_dub.duplicated(["month", "return_type"], keep=False)
dups = bengol_dub[mask].sort_values(["month", "return_type"])
print(dups.shape)
dups[["month", "return_type", "state_code", "eligible", "total_filed"]]

In [ ]:
g_copy.groupby("state_code")["state"].unique().loc[[97]]

In [ ]:
h = g[['state', 'state_code']].drop_duplicates()

# Optional: reset the index if needed
h = h.reset_index(drop=True)

In [ ]:
h

In [ ]:
state_97_rows = g_copy[g_copy['state_code'] == 97]
len(state_97_rows)

In [ ]:
state_97_rows = g_copy[(g_copy['state_code'] == 97)&(g_copy['state']=='DROP')]
len(state_97_rows)

In [ ]:
dropstate=g_copy[g_copy['state']=='DROP']
len(dropstate)

In [ ]:
g_clean = g_copy[g_copy["state_code"] != 97].copy()
print(len(g_clean))
print(g_clean["state"].nunique())
print(g_clean["state"].value_counts().head(3))

In [ ]:
print(g_clean["state"].value_counts())

In [ ]:
count_cols = ["eligible", "on_time", "late", "total_filed"]

g_merged = g_clean.groupby(["state", "month", "return_type"], as_index=False)[count_cols].sum(min_count=1)

print(len(g_merged))

In [ ]:
counts = g_merged["state"].value_counts()
print(counts.head())
print(counts.tail())
print((counts == 206).all())

In [ ]:
g_merged["date"] = pd.to_datetime(g_merged["month"], format="%B, %Y")


In [ ]:
g_merged["date"]

In [ ]:
g_merged["year"] = g_merged["date"].dt.year
g_merged["quarter"] = g_merged["date"].dt.quarter

In [ ]:
print(g_merged[["month", "year", "quarter"]].head())
print(g_merged["quarter"].unique())

In [ ]:
g_win = g_merged[g_merged["year"].between(2018, 2024)].copy()
print(len(g_win))

In [ ]:
g_q = (
    g_win.groupby(["state", "year", "quarter", "return_type"])[
        ["eligible", "on_time", "late", "total_filed"]
    ]
    .agg(lambda x: x.sum(min_count=1))
    .reset_index()
)

print(len(g_q))


In [ ]:
import numpy as np

# Step 1: Efficient monthly to quarterly aggregation
g_q = (
    g_win.groupby(["state", "year", "quarter", "return_type"])[
        ["eligible", "on_time", "late", "total_filed"]
    ]
    .sum(min_count=1)
    .reset_index()
)

# Step 2: Compute performance metrics and flag anomalies
g_q["filing_pct"] = (g_q["on_time"] / g_q["eligible"].replace(0, np.nan)) * 100
g_q["suspect_zero_eligible"] = (g_q["eligible"] == 0) & (g_q["total_filed"] > 0)

# Check outputs
print("Row count:", len(g_q))
print("\nFiling % Distribution:")
print(g_q["filing_pct"].describe())
print("\nSuspect Quarterly Rows:", g_q["suspect_zero_eligible"].sum())


In [ ]:
count_cols = ["eligible", "on_time", "late", "total_filed"]

# 1. Do any count columns hold negative values?
print((g_q[count_cols] < 0).sum())

# 2. Show the rows with a negative filing %
bad = g_q[g_q["filing_pct"] < 0]
print(len(bad))
bad[["state", "year", "quarter", "return_type"] + count_cols + ["filing_pct"]]

In [ ]:
import numpy as np

g_q["invalid_negative"] = g_q["on_time"] < 0
g_q.loc[g_q["invalid_negative"], ["on_time", "filing_pct"]] = np.nan

print(g_q["invalid_negative"].sum())
print(g_q["filing_pct"].describe())